In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import matplotlib.dates as mdates
from sklearn import datasets, linear_model
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.lines import Line2D
import statsmodels.api as sm
from scipy.stats import t
from scipy.optimize import minimize
import os
import cvxpy as cp
from tqdm import tqdm
import seaborn as sns
import torch

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import openpyxl

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.autograd import Function

from sklearn.preprocessing import StandardScaler

from functions import CVaRSolver, SPOPlus, VARasNN
import functions

# Data Preparation

Do following for different risk aversions beta:

for date in test_dates:

1. prepare train and test data (date)

2. prepare scenario matrix out of train data

3. specify constants for cvxpy (dimensions of scenario matrix, etc.)

4. precompute oracle solution w*(c)

5. Train VAR on train with custom loss function (combinations of MSE and DFL loss)

    Do so with every combination
    Save relevant metrics
    Estimate returns c_hat and w*(c_hat)

-> Agreggate metrics ofer the whole test period (backtesting period)

In [3]:
data_path = "../data/processed/"

return_data = pd.read_csv(data_path + "stationary_return_data_subset_15_random.csv")

In [4]:
# Pivot return data
return_matrix = return_data.pivot(index='date', columns='RIC', values='return')
display(return_matrix)

RIC,ADI.OQ,BKNG.OQ,CTAS.OQ,FDX.N,GWW.N,MAA.N,MDT.N,MSFT.OQ,NEM.N,NTRS.OQ,PXD.N,REGN.OQ,RTX.N,SNPS.OQ,WBA.OQ
date,,,,,,,,,,,,,,,
2000-01-31,0.005376,0.224274,-0.118235,-0.033588,0.002614,0.003109,0.256948,-0.161670,-0.168367,0.139151,-0.041958,-0.034314,-0.185577,-0.308052,-0.055556
2000-02-29,0.679144,-0.035560,-0.142458,-0.116904,-0.103902,0.033898,0.058743,-0.086845,0.085890,-0.064182,-0.029197,3.588832,-0.034088,-0.135318,-0.064469
2000-03-31,0.026274,0.430168,0.471831,0.116279,0.267153,-0.016393,0.061935,0.188811,0.015535,0.198793,0.278195,-0.476770,0.240491,0.220657,-0.002421
2000-04-28,-0.046548,-0.209375,0.011164,-0.033654,-0.200461,0.087511,0.010442,-0.343529,0.044568,-0.050879,-0.029412,-0.033827,-0.015826,-0.138462,0.092233
2000-05-31,0.002441,-0.397233,0.110410,-0.061360,-0.075723,-0.018325,-0.006017,-0.103047,-0.016000,0.026316,0.448485,-0.286652,-0.025159,0.126488,0.010094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-09-29,-0.032262,-0.006789,-0.045937,0.019990,-0.031226,-0.114164,-0.030282,-0.036643,-0.052920,-0.077564,-0.027916,-0.004271,-0.163529,0.000174,-0.121296
2023-10-31,-0.101434,-0.095459,0.054282,-0.093689,0.054912,-0.071856,-0.099541,0.070816,0.014073,-0.051382,0.041168,-0.052335,0.130888,0.022812,-0.052158
2023-11-30,0.165576,0.120499,0.093659,0.078009,0.079753,0.053576,0.123441,0.122944,0.083216,0.202397,-0.017303,0.056316,0.008468,0.157187,-0.031569


In [5]:
X, Y = functions.create_time_series_data_with_lags(return_matrix, max_lag=3)

print(X.shape)
print(Y.shape)

(286, 45)
(286, 15)


## Try looping

In [12]:
beta_levels = [0.07, 0.08, 0.09, 0.10] # feasible CVaR levels
beta_levels = [0.09]

n_test_months = 12
n_test_months = 4

gamma_levels = [0, 0.25, 0.5, 0.75, 1.0]
gamma_levels = [0.2]

n_epochs = 10            # HYPERPARAM: number of epochs

In [7]:
test_months = [5]

Problem: scenario loss marix is constant for all entries in X_train and in training it contains scenarios which are in the future.

Is it really a Problem?

I think its a feature: we trade of (very limited) data leakage in training for a richer scenario matrix which is actually the best possible for estimating the true risk at test date!


In [ ]:
results = []

for beta in beta_levels:
    
    # for test_index in range(1,n_test_months+1):
    for test_index in test_months:

        print(f"\nBeta = {beta}; Test Index = {test_index}\n")
        print("Preparing Data...")

        # data preparation
        X_train, X_val, X_test, Y_train, Y_val, Y_test = functions.create_train_test_split(X, Y, test_index, val_length=3)
        scenario_loss_matrix = functions.get_scenario_loss_matrix(return_matrix, test_index, num_scenarios=1000, random_seed=42)
        S, N = scenario_loss_matrix.shape # S scenarios × N assets

        # solver
        solver = CVaRSolver(
            loss_matrix=scenario_loss_matrix,
            N=N,
            S=S,
            alpha=0.95, # CVaR confidence level
            beta=beta
        )

        # Precompute oracle solutions
        oracle_solutions = [
            solver.solve(c = - mu).copy()
            for mu in tqdm(Y_train, desc="Computing oracle solutions")
        ]
        oracle_tensor = torch.tensor(
            np.array(oracle_solutions),
            dtype=torch.float32
        )

        # Prepare dataset for pytorch training
        x_scaler = StandardScaler()
        X_train = x_scaler.fit_transform(X_train)
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)
        train_dataset = TensorDataset(
            X_train_tensor,
            Y_train_tensor,
            oracle_tensor
        )
        batch_size = 8                                                          # HYPERPARAM: Batch size
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,        # TensorDataset shuffles all tensors together
            drop_last=True
        )

        # scale validation data using the training scaler
        X_val_scaled = x_scaler.transform(X_val)
        X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
        Y_val_tensor = torch.tensor(Y_val, dtype=torch.float32)

        # specify model for scale computation
        model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
        criterion = nn.MSELoss()
        # compute scale factor for the two losses to make them comparable
        spo_scale, mse_scale = functions.compute_loss_normalization(model, train_loader, solver, criterion)
        print(f"SPO+ scale: {spo_scale:.6f}", f"; MSE  scale: {mse_scale:.6f}\n")

        print("Train Models for different loss combinations:")
        
        for gamma in gamma_levels:
            
            print(f"\nGamma = {gamma} (weight of SPO+ loss)\n")

            # specify model and optimizer
            model = VARasNN(input_dim=X_train.shape[1], output_dim=Y_train.shape[1])
            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=1e-3                                                             # HYPERPARAM: Learning rate
            )
            criterion = nn.MSELoss()

            # Train model and retrieve training history
            history = functions.train_model(model,
                                            n_epochs,
                                            train_loader,
                                            optimizer,
                                            criterion,
                                            solver,
                                            spo_scale,
                                            mse_scale,
                                            gamma
                                            # val_X=X_val_tensor,
                                            # val_Y=Y_val_tensor,
                                            # early_stopping_patience=3
                                            )

            # Inference
            
            # Scale test data using training scaler
            X_test_scaled = x_scaler.transform(X_test.reshape(1, -1))

            X_test_tensor = torch.tensor(
                X_test_scaled,
                dtype=torch.float32
            )

            # Predict expected returns
            model.eval()
            with torch.no_grad():
                mu_hat_test = model(X_test_tensor)

            mu_hat_test = mu_hat_test.cpu().numpy()[0]

            test_mse = np.mean((mu_hat_test - Y_test)**2)

            # Compute portfolio weights from predicted returns
            w_hat = solver.solve(c = - mu_hat_test).copy()

            results.append({

                # id
                "run_id": f"b{beta}_g{gamma}_t{test_index}",

                # experiment settings
                "beta": beta,
                "gamma": gamma,
                "test_index": test_index,

                # dimensions
                "n_train": len(X_train),
                "n_test": len(X_test),
                "n_assets": N,
                "n_scenarios": S,

                # normalization factors
                "spo_scale": spo_scale,
                "mse_scale": mse_scale,

                # forecasting results
                "Y_hat_test": mu_hat_test,
                "Y_test": Y_test,
                "test_mse": test_mse,

                # portfolio results
                "weights": w_hat,

                # training history
                "history": history,
                "final_spo_loss": history["spo_loss"][-1],
                "final_mse_loss": history["mse_loss"][-1],
                "final_combined_loss": history["combined_loss"][-1],

                # model reproduction
                # "model_state_dict": model.state_dict()
            })


Beta = 0.09; Test Index = 5

Preparing Data...


Computing loss scales for normalization: 100%|██████████| 70/70 [00:30<00:00,  2.26it/s]


SPO+ scale: 1.003495 ; MSE  scale: 0.335045

Train Models for different loss combinations:

Gamma = 1.0 (weight of SPO+ loss)



epochs:  10%|█         | 1/10 [00:31<04:42, 31.44s/it]

Epoch 1: combined=0.908659 | SPO+ scaled=0.908659 | MSE scaled=0.895011


...

Epoch 8: combined=0.036624 | SPO+ scaled=0.162861 | MSE scaled=0.036624


epochs:  90%|█████████ | 9/10 [04:31<00:28, 28.39s/it]

Epoch 9: combined=0.031972 | SPO+ scaled=0.143190 | MSE scaled=0.031972


epochs: 100%|██████████| 10/10 [04:59<00:00, 29.94s/it]

Epoch 10: combined=0.029344 | SPO+ scaled=0.131561 | MSE scaled=0.029344


In [11]:
results[0]

{'run_id': 'b0.09_g1.0_t5',
 'beta': 0.09,
 'gamma': 1.0,
 'test_index': 5,
 'n_train': 281,
 'n_test': 45,
 'n_assets': 15,
 'n_scenarios': 1000,
 'spo_scale': np.float64(1.003494941815734),
 'mse_scale': np.float64(0.3350449393902506),
 'Y_hat_test': array([-0.01991093,  0.09030834,  0.21993953,  0.21901141,  0.07344068,
         0.12016219,  0.04482793,  0.19422388, -0.11767644,  0.09828539,
        -0.28557175,  0.0773604 ,  0.15047467,  0.12346746, -0.04534292],
       dtype=float32),
 'Y_test': array([-0.03226208, -0.00678898, -0.04593689,  0.01998985, -0.03122637,
        -0.11416374, -0.03028232, -0.03664267, -0.05291994, -0.07756353,
        -0.02791602, -0.00427107, -0.16352859,  0.00017433, -0.12129593]),
 'test_mse': np.float64(0.03149463752299389),
 'weights': array([-0.        ,  0.0242342 ,  0.2       ,  0.01209378, -0.        ,
         0.18532092,  0.1102714 ,  0.2       , -0.        , -0.        ,
        -0.        ,  0.07658935,  0.19149035, -0.        , -0.        

In [9]:
oracle_array = np.array(oracle_solutions)
unique = np.unique(oracle_array.round(4), axis=0)
print(f"Unique oracle solutions: {len(unique)} out of {len(oracle_solutions)}")

Unique oracle solutions: 281 out of 281


## XX

12 months x 5 gamma levels x 4 risk aversion levels x 3 min training = 720min = 12h

In [8]:
len(Y_train)

283

In [9]:
unique_portfolios = set()

for mu in Y_train:
    w = solver.solve(-mu)
    unique_portfolios.add(tuple(np.round(w, 3)))

print(len(unique_portfolios))

268


In [10]:
unique_portfolios

{(np.float64(-0.0),
  np.float64(-0.0),
  np.float64(-0.0),
  np.float64(-0.0),
  np.float64(0.183),
  np.float64(0.188),
  np.float64(0.114),
  np.float64(0.2),
  np.float64(0.152),
  np.float64(0.162)),
 (np.float64(-0.0),
  np.float64(-0.0),
  np.float64(-0.0),
  np.float64(-0.0),
  np.float64(0.2),
  np.float64(0.143),
  np.float64(0.2),
  np.float64(0.196),
  np.float64(0.061),
  np.float64(0.2)),
 (np.float64(-0.0),
  np.float64(-0.0),
  np.float64(-0.0),
  np.float64(0.041),
  np.float64(0.2),
  np.float64(0.077),
  np.float64(0.2),
  np.float64(0.164),
  np.float64(0.118),
  np.float64(0.2)),
 (np.float64(-0.0),
  np.float64(-0.0),
  np.float64(-0.0),
  np.float64(0.052),
  np.float64(0.2),
  np.float64(0.163),
  np.float64(0.2),
  np.float64(0.185),
  np.float64(-0.0),
  np.float64(0.2)),
 (np.float64(-0.0),
  np.float64(-0.0),
  np.float64(0.033),
  np.float64(-0.0),
  np.float64(0.2),
  np.float64(0.2),
  np.float64(0.2),
  np.float64(0.167),
  np.float64(-0.0),
  np.float64